# 44. Local, Sliding-Window and Sparse Attention | 局部、滑动窗口与稀疏注意力

**难度：** Medium | **环境：** CPU-first | **标签：** `Attention`, `长上下文`, `稀疏访问`

## 本节导读

标准 Attention 让每个 token 都可以读取全部历史位置，因此能够建立全局依赖，但序列变长时，计算与中间分数矩阵也会快速增长。局部、滑动窗口和块稀疏 Attention 的共同思想不是改变 Q/K/V 的语义，而是改变每个 Query 被允许访问哪些 Key。

本节比较三类访问模式如何在全局信息、理论连接数和长上下文成本之间取舍，并用 mask 验证“允许访问什么”这一核心机制。它为后续架构演进、长上下文推理和稀疏算子实现提供共同的访问模式语言。

**关键词：** `Dense Attention`, `Sliding Window`, `Block Sparse`, `Access Mask`


## 前置阅读

**导语：** 先理解 Q/K/V、因果 mask 与多头形状，再比较不同访问范围怎样限制每个 Query 可读取的历史位置。

- [Part 02 · 04 Attention（MHA / GQA）](./04_Attention_MHA_GQA.ipynb)
- [大模型架构 · Attention 演化](../topic_discussion/llm_architecture_evolution/04_attention_evolution.md)


### Step 1：Attention 的访问范围

同一段 Q/K/V 可以搭配不同的访问规则。Dense causal 保留完整历史；滑动窗口保留最近邻域；规则块稀疏在局部连接之外保留少量远程捷径。它们首先改变当前 Query 能直接读取哪些历史 Key，随后才改变连接账本、状态规模和实现路径。

![Dense、滑动窗口与块稀疏 Attention 的访问范围](../docs/public/02_PyTorch_Algorithms/44_attention_access_patterns.svg)


### Step 2：从访问规则到理论连接数

设序列长度为 `L`，窗口宽度为 `w`。自回归模式始终要求 `key ≤ query`：Dense 保留完整因果三角；滑动窗口只保留最近 `w` 个历史位置；规则块稀疏先选择 block 对，再在 block 内过滤未来位置；选择性稀疏则由内容或检索结果决定远程连接。

| 访问模式 | 每个 Query 可读取的位置 | 因果连接数 | 主要风险 |
| --- | --- | --- | --- |
| Dense causal | 全部历史与当前位置 | `L(L+1)/2` | 长序列下分数矩阵和访存增长快 |
| Sliding window | 最近 `min(w, t+1)` 个位置 | `wL - w(w-1)/2`（`L ≥ w`） | 超出窗口的远距离依赖不能单层直达 |
| Rule-based block sparse | 预先指定的局部或远程 block 对 | 由激活 block 对及因果过滤决定 | 非规则模式可能难以映射到高效 kernel |
| Selective sparse | 内容或检索选择的位置 | 与选择数量和命中率有关 | 选择错误会遗漏关键上下文 |

连接数给出的是结构账本，不是速度承诺：实际收益还取决于 shape、mask 表示、kernel 与 backend 能否利用这一规则。


### Step 3：长上下文中的可达性与取舍

局部窗口减少的是**单层直接访问范围**，并不等于远距离信息永远无法影响当前位置：若每层都允许向前看 `w-1` 个位置，堆叠多层后信息可以逐层传播。对距离为 `d` 的两个 token，覆盖它至少需要 `ceil(d / (w-1))` 层；这条路径越长，跨位置的信息传递越依赖模型在中间层保留有效表示。块稀疏或选择性稀疏可以增加远程捷径，但前提是被激活的 block 或位置确实包含任务需要的信息。

连接数或 mask 密度只能说明“允许看什么”。比较长上下文方案时，还应在训练长度内外分别记录任务质量，并在固定模型、shape 和 backend 下测量延迟与显存。

| 观察角度 | Dense | 局部 / 滑动窗口 | 块 / 选择性稀疏 |
| --- | --- | --- | --- |
| 单层直接依赖 | 完整历史 | 最近 `w` 个位置 | 指定或动态选择的位置 |
| 远程信息到达当前 token | 直接读取 | 经过多层局部传播 | 依赖远程 block / 位置是否被选中 |
| 连接与状态规模 | 随完整三角增长 | 随窗口缩小 | 取决于激活 block 或选择数 |
| 后续证据 | 长依赖任务质量、延迟、显存 | 距离覆盖、长依赖质量、延迟 | 选择命中率、长依赖质量、规则稀疏 kernel 证据 |

![局部窗口通过层堆叠传播，稀疏捷径缩短远程路径](../docs/public/02_PyTorch_Algorithms/44_sparse_reachability.svg)


### Step 4：实现访问 mask、连接账本与传播层数

题目区把访问策略拆成四个可验证的机制：局部因果可见性、block 对可见性、连接账本，以及局部窗口下远距离信息需要经过多少层传播。骨架提供输入检查；输出均为 CPU 机制账本，后续 GPU 单元才记录当前 SDPA 实现路径的成本。

| TODO | 代码契约与机制责任 | 关键约束 | 测试证据 |
| --- | --- | --- | --- |
| TODO 1 | 构造局部因果 mask，允许当前位置读取窗口内历史 | `seq_len`、`window_size` 为正整数；不得泄漏未来 | 正常窗口、首 token、未来与窗口外历史 |
| TODO 2 | 按激活 block 对构造稀疏 mask，并按需施加因果过滤 | block 坐标不得越界；因果模式拒绝未来 block | 合法 / 空 / 越界 block，对未来 block 的处理 |
| TODO 3 | 汇总可见连接数、总连接数与密度 | 输入必须是方形 `bool` mask | Dense、局部与 block 稀疏账本对照 |
| TODO 4 | 估算局部窗口中跨距离信息的最少传播层数 | `distance >= 0`；`window_size=1` 时非零距离不可达 | 相邻、远距离、不可达窗口与非法输入 |


In [ ]:
import numpy as np


In [ ]:
def build_local_causal_mask(seq_len: int, window_size: int) -> np.ndarray:
    """返回 [query, key] 布尔 mask；True 表示当前位置允许读取该 key。"""
    if not isinstance(seq_len, int) or not isinstance(window_size, int) or seq_len <= 0 or window_size <= 0:
        raise ValueError('seq_len 和 window_size 必须为正整数')
    mask = np.zeros((seq_len, seq_len), dtype=bool)
    # TODO 1：局部因果访问；query 只能读取 key <= query 且 query-key < window_size。
    # mask = ???
    return mask

def build_block_sparse_mask(
    seq_len: int, block_size: int, active_block_pairs: set[tuple[int, int]], causal: bool = True
) -> np.ndarray:
    """按 (query_block, key_block) 构造 block 稀疏 mask；causal=True 时屏蔽未来位置。"""
    if seq_len <= 0 or block_size <= 0:
        raise ValueError('seq_len 和 block_size 必须为正整数')
    num_blocks = (seq_len + block_size - 1) // block_size
    mask = np.zeros((seq_len, seq_len), dtype=bool)
    for query_block, key_block in active_block_pairs:
        if not (0 <= query_block < num_blocks and 0 <= key_block < num_blocks):
            raise ValueError('active_block_pairs 包含越界 block')
        if causal and key_block > query_block:
            raise ValueError('因果 block 稀疏不能激活未来 block')
        q_start, q_end = query_block * block_size, min((query_block + 1) * block_size, seq_len)
        k_start, k_end = key_block * block_size, min((key_block + 1) * block_size, seq_len)
        # TODO 2：激活当前 block 对覆盖的 query/key 区域。
        # mask[q_start:q_end, k_start:k_end] = ???
    if causal:
        mask &= np.tri(seq_len, dtype=bool)
    return mask

def summarize_access(mask: np.ndarray) -> dict[str, int | float]:
    """统计可见连接数与相对密度，不把理论稀疏度写成真实 kernel 加速。"""
    if mask.ndim != 2 or mask.shape[0] != mask.shape[1] or mask.dtype != bool:
        raise ValueError('mask 必须是方形 bool 数组')
    # TODO 3：计算 visible_pairs、total_pairs 与 density。
    # visible_pairs = ???
    # total_pairs = ???
    # density = ???
    return {'visible_pairs': visible_pairs, 'total_pairs': total_pairs, 'density': density}


def min_local_propagation_layers(distance: int, window_size: int) -> int | float:
    """返回局部窗口把距离为 distance 的历史信息传到当前位置所需的最少层数。

    每层最多向前传播 window_size - 1 个位置；返回 float('inf') 表示窗口为 1 时无法跨位置传播。
    """
    if not isinstance(distance, int) or not isinstance(window_size, int) or distance < 0 or window_size <= 0:
        raise ValueError('distance 必须为非负整数，window_size 必须为正整数')
    # TODO 4：根据每层最大传播步长计算最少层数。
    # step_per_layer = window_size - 1
    # layers = ???
    return layers


In [ ]:
# 测试访问不变量：因果窗口不能泄漏未来，block 对必须精确生效，账本只统计 mask 中的连接。
def test_local_causal_visibility():
    """验证窗口内历史可见，未来和窗口外历史不可见。"""
    mask = build_local_causal_mask(5, 2)
    assert mask[3, 3] and mask[3, 2]
    assert not mask[3, 1] and not mask[2, 3]

def test_block_sparse_pairs_and_bounds():
    """验证指定 block 对与越界输入。"""
    mask = build_block_sparse_mask(4, 2, {(0, 0), (1, 0)})
    assert mask[1, 0] and mask[3, 0] and not mask[0, 1] and not mask[0, 3]
    assert not build_block_sparse_mask(4, 2, set()).any()
    try:
        build_block_sparse_mask(4, 2, {(2, 0)})
    except ValueError:
        return
    raise AssertionError('越界 block 必须拒绝')

def test_block_sparse_causal_filter():
    """验证因果模式拒绝未来 block；双向模式可显式保留它。"""
    try:
        build_block_sparse_mask(4, 2, {(0, 1)})
    except ValueError:
        pass
    else:
        raise AssertionError('因果 block 稀疏不应激活未来 block')
    bidirectional = build_block_sparse_mask(4, 2, {(0, 1)}, causal=False)
    assert bidirectional[0, 2]

def test_access_summary():
    """验证连接数和密度与 mask 一致。"""
    summary = summarize_access(build_local_causal_mask(4, 2))
    assert summary == {'visible_pairs': 7, 'total_pairs': 16, 'density': 7 / 16}

test_local_causal_visibility()
test_block_sparse_pairs_and_bounds()
test_block_sparse_causal_filter()
def test_local_propagation_layers():
    """验证局部窗口的多层传播范围与不可达边界。"""
    assert min_local_propagation_layers(0, 1) == 0
    assert min_local_propagation_layers(1, 2) == 1
    assert min_local_propagation_layers(5, 3) == 3
    assert min_local_propagation_layers(1, 1) == float('inf')
    try:
        min_local_propagation_layers(-1, 2)
    except ValueError:
        return
    raise AssertionError('负距离必须拒绝')

test_access_summary()
test_local_propagation_layers()
print('✅ 局部与稀疏 Attention 访问机制测试通过。')


---

## 参考代码与解析


In [ ]:
def build_local_causal_mask(seq_len: int, window_size: int) -> np.ndarray:
    """返回 [query, key] 布尔 mask；True 表示当前位置允许读取该 key。"""
    if not isinstance(seq_len, int) or not isinstance(window_size, int) or seq_len <= 0 or window_size <= 0:
        raise ValueError('seq_len 和 window_size 必须为正整数')
    # TODO 1：局部因果访问。
    query, key = np.indices((seq_len, seq_len))
    mask = (key <= query) & ((query - key) < window_size)
    return mask

def build_block_sparse_mask(
    seq_len: int, block_size: int, active_block_pairs: set[tuple[int, int]], causal: bool = True
) -> np.ndarray:
    """按 (query_block, key_block) 构造 block 稀疏 mask；causal=True 时屏蔽未来位置。"""
    if seq_len <= 0 or block_size <= 0:
        raise ValueError('seq_len 和 block_size 必须为正整数')
    num_blocks = (seq_len + block_size - 1) // block_size
    mask = np.zeros((seq_len, seq_len), dtype=bool)
    # TODO 2：激活指定 block 对，并在因果模式下拒绝未来 block。
    for query_block, key_block in active_block_pairs:
        if not (0 <= query_block < num_blocks and 0 <= key_block < num_blocks):
            raise ValueError('active_block_pairs 包含越界 block')
        if causal and key_block > query_block:
            raise ValueError('因果 block 稀疏不能激活未来 block')
        q_start, q_end = query_block * block_size, min((query_block + 1) * block_size, seq_len)
        k_start, k_end = key_block * block_size, min((key_block + 1) * block_size, seq_len)
        # TODO 2：激活当前 block 对覆盖的 query/key 区域。
        mask[q_start:q_end, k_start:k_end] = True
    if causal:
        mask &= np.tri(seq_len, dtype=bool)
    return mask

def summarize_access(mask: np.ndarray) -> dict[str, int | float]:
    """统计可见连接数与相对密度，不把理论稀疏度写成真实 kernel 加速。"""
    if mask.ndim != 2 or mask.shape[0] != mask.shape[1] or mask.dtype != bool:
        raise ValueError('mask 必须是方形 bool 数组')
    # TODO 3：连接账本。
    visible_pairs = int(mask.sum())
    total_pairs = mask.size
    density = visible_pairs / total_pairs
    return {'visible_pairs': visible_pairs, 'total_pairs': total_pairs, 'density': density}


def min_local_propagation_layers(distance: int, window_size: int) -> int | float:
    """返回局部窗口把距离为 distance 的历史信息传到当前位置所需的最少层数。

    每层最多向前传播 window_size - 1 个位置；返回 float('inf') 表示窗口为 1 时无法跨位置传播。
    """
    if not isinstance(distance, int) or not isinstance(window_size, int) or distance < 0 or window_size <= 0:
        raise ValueError('distance 必须为非负整数，window_size 必须为正整数')
    # TODO 4：每层可覆盖 window_size - 1 个新的历史位置。
    step_per_layer = window_size - 1
    if distance == 0:
        return 0
    if window_size == 1:
        return float('inf')
    return (distance + step_per_layer - 1) // step_per_layer


### 答案解析

- **TODO 1：** 局部因果可见性同时满足 `key <= query` 与 `query-key < window_size`；只保留其中一条都会错误地泄漏未来或窗口外历史。
- **TODO 2：** block 对先决定候选区域，再根据 `causal` 过滤未来位置。因此同一组 block 在自回归和双向任务中不能直接复用。
- **TODO 3：** `density` 只描述理论连接比例；要判断收益还要固定模型、shape 与 backend，观察延迟、显存和任务质量。
- **TODO 4：** 局部窗口每层最多让信息向前传播 `window_size - 1` 个位置。传播层数解释了局部 Attention 如何间接覆盖远距离，但不能替代真实长依赖任务上的质量验证。


### Step 5：可选 GPU 复测——Dense 与滑动窗口的 SDPA 成本探针

这一组单元在相同随机 Q/K/V、dtype、head 数与序列长度下，比较 Dense causal SDPA 和**显式布尔滑动窗口 mask** 的 prefill 调用。它记录当前 PyTorch 实现路径的延迟、峰值显存、mask 存储量、可见连接数，以及末位置能否直接读取最早 token。随机输入不代表长文问答、检索或长度外推质量。

#### 5.1 环境与实验配置

| 字段 | 本次设置 | 作用 |
| --- | --- | --- |
| `seq_lens` | 一组递增序列长度 | 观察连接规模扩大时的局部趋势 |
| `window_size` | 固定局部窗口 | 候选路径每个 Query 的直接历史范围 |
| `batch_size / heads / head_dim` | 两条路径共用 | 避免把张量形状差异当成策略收益 |
| `warmup / repeats` | 预热后重复测量 | 降低首次调用与瞬时波动影响 |
| `RUN_GPU_EXPERIMENT` | 默认 `0` | 设为 `1` 才执行 GPU 并写入 JSON |


In [ ]:
# GPU 配置：默认关闭；设置 RUN_GPU_EXPERIMENT=1 后才会运行并写入结果。
import json
import os
from datetime import datetime, timezone
from pathlib import Path


def resolve_tutorial_root() -> Path:
    """解析本地仓库或标准 Colab 克隆目录。"""
    configured = os.environ.get("LLM_ALGO_REPO_DIR")
    candidates = [Path(configured).expanduser()] if configured else []
    candidates.extend([Path.cwd(), Path("/content/llm-algo-leetcode")])
    for candidate in candidates:
        if (candidate / "02_PyTorch_Algorithms").is_dir():
            return candidate
    raise RuntimeError("未找到教程根目录；请在仓库根目录运行，或设置 LLM_ALGO_REPO_DIR。")


RUN_GPU_EXPERIMENT = os.environ.get("RUN_GPU_EXPERIMENT", "0") == "1"
REPO_ROOT = resolve_tutorial_root()
GPU_RESULTS_DIR = REPO_ROOT / "benchmarks" / "results" / "44_sparse_attention"
GPU_CONFIG = {
    "batch_size": 1,
    "heads": 8,
    "head_dim": 64,
    "seq_lens": [512, 1024],
    "window_size": 256,
    "dtype": "float16",
    "warmup": 5,
    "repeats": 20,
    "mask_representation": "explicit_boolean_mask",
}
print("GPU probe is", "enabled" if RUN_GPU_EXPERIMENT else "disabled")
print("Set RUN_GPU_EXPERIMENT=1 to run the SDPA probe.")


#### 5.2 执行对照并保存 JSON

Dense 路径使用 PyTorch SDPA 的因果模式；候选路径使用同一份 Q/K/V 和显式布尔滑动窗口 mask。这个 mask 本身是 `L×L` 张量，可能改变 SDPA backend 或增加显存，因此结果只用于判断当前实现路径是否值得继续优化，不等同于规则稀疏 kernel 的性能结论。

In [ ]:
# 仅在显式开启后运行：PyTorch SDPA 的 Dense / 滑动窗口局部成本探针。
if not RUN_GPU_EXPERIMENT:
    print("GPU probe skipped. Set RUN_GPU_EXPERIMENT=1 to execute it.")
else:
    import torch
    import torch.nn.functional as F

    if not torch.cuda.is_available():
        raise RuntimeError("RUN_GPU_EXPERIMENT=1 requires a CUDA-enabled PyTorch runtime.")

    device = torch.device("cuda")
    dtype = getattr(torch, GPU_CONFIG["dtype"])

    def sliding_window_mask(seq_len: int, window_size: int, device: torch.device) -> torch.Tensor:
        """返回 SDPA 布尔 mask；True 表示当前 Query 可以读取该 Key。"""
        positions = torch.arange(seq_len, device=device)
        query = positions[:, None]
        key = positions[None, :]
        return (key <= query) & ((query - key) < window_size)

    def measure_sdpa(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, *, attn_mask=None, is_causal=False) -> dict:
        """预热后测量固定 SDPA workload；结果只代表当前本地实现路径。"""
        for _ in range(GPU_CONFIG["warmup"]):
            F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, is_causal=is_causal)
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)
        start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(GPU_CONFIG["repeats"]):
            output = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, is_causal=is_causal)
        end.record()
        torch.cuda.synchronize(device)
        return {
            "mean_ms": start.elapsed_time(end) / GPU_CONFIG["repeats"],
            "peak_allocated_mb": torch.cuda.max_memory_allocated(device) / 2**20,
            "output_finite": bool(torch.isfinite(output).all().item()),
        }

    rows = []
    for seq_len in GPU_CONFIG["seq_lens"]:
        torch.manual_seed(44 + seq_len)
        q = torch.randn(GPU_CONFIG["batch_size"], GPU_CONFIG["heads"], seq_len, GPU_CONFIG["head_dim"], device=device, dtype=dtype)
        k = torch.randn_like(q)
        v = torch.randn_like(q)
        window_mask = sliding_window_mask(seq_len, GPU_CONFIG["window_size"], device)
        baseline = measure_sdpa(q, k, v, is_causal=True)
        candidate = measure_sdpa(q, k, v, attn_mask=window_mask)
        rows.append({
            "seq_len": seq_len,
            "dense_visible_pairs": seq_len * (seq_len + 1) // 2,
            "window_visible_pairs": int(window_mask.sum().item()),
            "mask_storage_mb": window_mask.numel() * window_mask.element_size() / 2**20,
            "last_token_can_read_first": bool(window_mask[-1, 0].item()),
            "baseline": baseline,
            "candidate": candidate,
        })

    faster_without_more_memory = all(
        row["candidate"]["mean_ms"] <= row["baseline"]["mean_ms"]
        and row["candidate"]["peak_allocated_mb"] <= row["baseline"]["peak_allocated_mb"]
        for row in rows
    )
    decision = "continue_to_model_quality_eval" if faster_without_more_memory else "investigate_mask_backend"
    decision_reason = (
        "窗口 mask 在所有长度上未增加峰值显存且不慢于 Dense；下一步可用真实模型检查长依赖质量。"
        if faster_without_more_memory
        else "显式布尔 mask 没有同时改善时间和峰值显存；先检查 backend、mask 表示与 shape。"
    )
    GPU_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    RESULT_PATH = GPU_RESULTS_DIR / f"gpu_sdpa_window_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.json"
    record = {
        "workload": GPU_CONFIG,
        "hardware": {"gpu": torch.cuda.get_device_name(device), "torch": torch.__version__, "cuda": torch.version.cuda},
        "backend_note": {
            "flash_sdp_enabled": torch.backends.cuda.flash_sdp_enabled(),
            "mem_efficient_sdp_enabled": torch.backends.cuda.mem_efficient_sdp_enabled(),
            "math_sdp_enabled": torch.backends.cuda.math_sdp_enabled(),
            "caveat": "这些开关不保证本次调用实际选中的 backend；显式 attn_mask 可能改变执行路径。",
        },
        "rows": rows,
        "failure": None,
        "evidence_level": "single_gpu_explicit_mask_sdpa_microbenchmark",
        "decision": decision,
        "decision_reason": decision_reason,
        "note": "Random Q/K/V verify access and local SDPA cost only; they do not measure long-context task quality.",
    }
    RESULT_PATH.write_text(json.dumps(record, indent=2), encoding="utf-8")
    print(f"Saved GPU probe: {RESULT_PATH}")


#### 5.3 读取结果并形成下一步判断

先核对两条路径的 shape 条件一致、输出均为有限值，再查看连接数、显式 mask 的存储量、平均延迟与峰值显存。`continue_to_model_quality_eval` 只表示当前实现路径值得进入真实模型验证；`investigate_mask_backend` 则表示应先检查 backend、mask 表示和 shape。

In [ ]:
# 读取最近一次探针并显示可比较字段；未运行时给出明确提示。
result_path = globals().get("RESULT_PATH")
if result_path is None:
    candidates = sorted(GPU_RESULTS_DIR.glob("gpu_sdpa_window_*.json"))
    if not candidates:
        print("No GPU result found. Run 5.2 with RUN_GPU_EXPERIMENT=1 first.")
    else:
        result_path = candidates[-1]
if result_path is not None:
    record = json.loads(Path(result_path).read_text(encoding="utf-8"))
    print(f"Result: {result_path}")
    print(f"Evidence: {record['evidence_level']} | GPU: {record['hardware']['gpu']}")
    print(f"Decision: {record['decision']} — {record['decision_reason']}")
    for row in record["rows"]:
        base, cand = row["baseline"], row["candidate"]
        speed_ratio = base["mean_ms"] / cand["mean_ms"] if cand["mean_ms"] else float("nan")
        print(
            f"L={row['seq_len']}: pairs {row['dense_visible_pairs']} -> {row['window_visible_pairs']}; "
            f"direct-first={row['last_token_can_read_first']}; mask MB={row['mask_storage_mb']:.2f}; "
            f"ms {base['mean_ms']:.3f} -> {cand['mean_ms']:.3f} (ratio={speed_ratio:.2f}); "
            f"peak MB {base['peak_allocated_mb']:.1f} -> {cand['peak_allocated_mb']:.1f}"
        )


## 相关阅读

- [Longformer 论文](https://arxiv.org/abs/2004.05150)
- [BigBird 论文](https://arxiv.org/abs/2007.14062)
- [大模型架构 · Attention 演化](../topic_discussion/llm_architecture_evolution/04_attention_evolution.md)
- [Part 02 · 20 FlashAttention 模拟与 SDPA 对照](./20_FlashAttention_Sim.ipynb)
